In [ ]:
import gurobipy as gb
import numpy as np

# Understanding CVaR

A minimal toy model to build intuition for **Conditional Value-at-Risk (CVaR)**: how it changes a decision compared to plain expected-value optimization, and how that shows up in the distribution of outcomes.

Story: we choose how much to `invest`, `x`, given 11 possible price realizations (scenarios), each equally likely. Profit in scenario `s` is `profit_s = x * price_s`.

We compare two decision rules:
1. Risk-neutral: maximize expected profit only.
2. CVaR-averse: maximize expected profit, but require the average profit in the worst `(1 - alpha)` share of scenarios (the CVaR) to stay above a risk budget `C`.

In [ ]:
# let's imagine 11 scenarios
scenarios = np.array(
    [
        -20,
        -10,
        0,
        12,
        20,
        30,
        40,
        50,
        100,
        110,
        120,
    ]
)
n_scenarios = len(scenarios)
probability = 1 / n_scenarios

# Case A: risk-neutral decision (no CVaR), maximize expected profit only
m = gb.Model()
x = m.addVar(name="investment", lb=0, ub=10)

m.setObjective((x * scenarios * probability).sum(), sense=gb.GRB.MAXIMIZE)
m.optimize()

x_A = x.X
profit_A = x_A * scenarios  # profit realized in each scenario, no probability weighting

print(f"Optimal investment (no CVaR): x* = {x_A}")
print(f"Expected profit: {profit_A.mean():.2f}")
print(f"Worst-case profit: {profit_A.min():.2f}")

## Case A: profit distribution without CVaR

Since the expected price is positive, the risk-neutral optimizer takes the maximum possible investment (`x* = 10`), even though some scenarios lose money. Let's plot the resulting profit across all 11 scenarios.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#e6e5e1"
plt.rcParams["grid.linewidth"] = 0.8
plt.rcParams["axes.axisbelow"] = True

BLUE = "#2a78d6"   # gain
RED = "#e34948"    # loss
ORANGE = "#eb6834"  # used later to identify the CVaR case
GRAY = "#52514e"

labels = [f"{p:g}" for p in scenarios]


def bar_plot(ax, profit, title, invest_x):
    colors = [BLUE if p >= 0 else RED for p in profit]
    ax.bar(labels, profit, color=colors, width=0.62, zorder=3)
    mean_profit = profit.mean()
    ax.axhline(mean_profit, color=GRAY, linestyle="--", linewidth=1.2, zorder=4)
    ax.text(0, mean_profit, f" E[profit]={mean_profit:,.0f}", va="bottom", ha="left",
            fontsize=9, color=GRAY)
    ax.axhline(0, color="#0b0b0b", linewidth=0.8, zorder=2)
    ax.set_title(f"{title}\ninvestment x* = {invest_x:.2f}", fontsize=11, loc="left")
    ax.set_ylabel("Profit")
    ax.set_xlabel("Price scenario")
    ax.tick_params(axis="x", labelrotation=45)


fig, ax = plt.subplots(figsize=(8, 4.5))
bar_plot(ax, profit_A, "Without CVaR (risk-neutral)", x_A)
fig.tight_layout()
plt.show()

## Adding CVaR

We use the standard Rockafellar-Uryasev linear formulation. Define the loss in scenario `s` as `L_s = -profit_s(x)`. Then, for confidence level `alpha`:

```
CVaR_alpha(L) = zeta + 1/(1 - alpha) * sum_s( probability_s * u_s )
u_s >= L_s - zeta
u_s >= 0
```

`zeta` and `u_s` are auxiliary decision variables that the solver picks optimally; `zeta` ends up being the Value-at-Risk (the loss threshold that separates the tail), and `u_s` measures how far scenario `s` falls below that threshold.

Case B keeps the same objective (maximize expected profit) but adds a constraint: `CVaR_alpha(Loss) <= C`, a risk budget on the average loss in the worst `(1 - alpha)` share of scenarios. We use `alpha = 0.8` (worst 20% of scenarios) and `C = 60`.

In [ ]:
alpha = 0.8
C = 60.0

mB = gb.Model("cvar_constrained")
xB = mB.addVar(name="investment", lb=0, ub=10)
zeta = mB.addVar(lb=-gb.GRB.INFINITY, name="VaR_proxy")
u = mB.addVars(n_scenarios, lb=0, name="tail_excess")

loss = [-(xB * scenarios[s]) for s in range(n_scenarios)]
mB.addConstrs((u[s] >= loss[s] - zeta for s in range(n_scenarios)), name="cvar_tail")
cvar_expr = zeta + (1 / (1 - alpha)) * gb.quicksum(probability * u[s] for s in range(n_scenarios))
mB.addConstr(cvar_expr <= C, name="risk_budget")

mB.setObjective((xB * scenarios * probability).sum(), sense=gb.GRB.MAXIMIZE)
mB.optimize()

x_B = xB.X
profit_B = x_B * scenarios

print(f"Optimal investment (with CVaR): x* = {x_B:.2f}")
print(f"Expected profit: {profit_B.mean():.2f}")
print(f"Worst-case profit: {profit_B.min():.2f}")
print(f"CVaR_{alpha:.0%}(loss): {cvar_expr.getValue():.2f}")

## Comparing the two distributions

Same scenarios, same y-axis, side by side: notice how the CVaR constraint pulls the investment down, shrinking both the downside tail and the upside.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
bar_plot(axes[0], profit_A, "Without CVaR", x_A)
bar_plot(axes[1], profit_B, f"With CVaR (alpha={alpha:.0%}, budget={C:.0f})", x_B)
fig.suptitle("Profit distribution across price scenarios", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# Same comparison, but paired scenario by scenario, to see the distribution move
fig, ax = plt.subplots(figsize=(9, 4.5))
idx = np.arange(n_scenarios)
w = 0.36
ax.bar(idx - w / 2, profit_A, width=w, color=BLUE, label="Without CVaR", zorder=3)
ax.bar(idx + w / 2, profit_B, width=w, color=ORANGE, label="With CVaR", zorder=3)
ax.axhline(0, color="#0b0b0b", linewidth=0.8, zorder=2)
ax.set_xticks(idx)
ax.set_xticklabels(labels, rotation=45)
ax.set_ylabel("Profit")
ax.set_xlabel("Price scenario")
ax.set_title("How the profit distribution moves once CVaR is added", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()

## Efficient frontier: expected profit vs. risk aversion

Instead of picking one risk budget `C`, we can sweep the risk aversion level and trace out the full trade-off. Define `beta` in `[0, 1]`:

- `beta = 0`: no constraint, reproduces the risk-neutral case (Case A)
- `beta = 1`: fully risk-averse, tightest feasible budget (`x* = 0`, no exposure at all)

The risk budget scales linearly between the two: `C(beta) = (1 - beta) * C_loose`, where `C_loose` is the CVaR already implied by the risk-neutral solution (Case A).

In [ ]:
def cvar_of(profit, alpha):
    """Exact CVaR_alpha of a fixed profit realization, via the same LP used above."""
    loss = -profit
    mc = gb.Model()
    mc.Params.OutputFlag = 0
    zeta_c = mc.addVar(lb=-gb.GRB.INFINITY)
    u_c = mc.addVars(n_scenarios, lb=0)
    mc.addConstrs((u_c[s] >= loss[s] - zeta_c for s in range(n_scenarios)))
    obj = zeta_c + (1 / (1 - alpha)) * gb.quicksum(probability * u_c[s] for s in range(n_scenarios))
    mc.setObjective(obj, sense=gb.GRB.MINIMIZE)
    mc.optimize()
    return obj.getValue()


C_loose = cvar_of(profit_A, alpha)  # CVaR implied by the risk-neutral solution
C_tight = 0.0  # CVaR at x = 0 (no exposure, no loss, no gain)

betas = np.linspace(0, 1, 21)
frontier_investment = np.zeros_like(betas)
frontier_profit = np.zeros_like(betas)

for i, beta in enumerate(betas):
    C_beta = (1 - beta) * C_loose + beta * C_tight

    mf = gb.Model()
    mf.Params.OutputFlag = 0
    xf = mf.addVar(name="investment", lb=0, ub=10)
    zetaf = mf.addVar(lb=-gb.GRB.INFINITY)
    uf = mf.addVars(n_scenarios, lb=0)
    lossf = [-(xf * scenarios[s]) for s in range(n_scenarios)]
    mf.addConstrs((uf[s] >= lossf[s] - zetaf for s in range(n_scenarios)))
    cvarf = zetaf + (1 / (1 - alpha)) * gb.quicksum(probability * uf[s] for s in range(n_scenarios))
    mf.addConstr(cvarf <= C_beta)
    mf.setObjective((xf * scenarios * probability).sum(), sense=gb.GRB.MAXIMIZE)
    mf.optimize()

    frontier_investment[i] = xf.X
    frontier_profit[i] = xf.X * scenarios.mean()

beta_B = 1 - C / C_loose  # where Case B (C = 60) sits on this same sweep
print(f"C_loose = {C_loose:.2f}")
print(f"Case B (C = {C:.0f}) corresponds to beta = {beta_B:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(betas, frontier_profit, color=GRAY, linewidth=1.6, zorder=2)
ax.scatter(betas, frontier_profit, color=GRAY, s=18, zorder=3)
ax.scatter([0], [profit_A.mean()], color=BLUE, s=90, zorder=4, label="Without CVaR")
ax.scatter([beta_B], [profit_B.mean()], color=ORANGE, s=90, zorder=4, label="With CVaR (budget=60)")
ax.set_xlabel("Risk aversion, beta (0 = risk-neutral, 1 = fully risk-averse)")
ax.set_ylabel("Expected profit")
ax.set_title("Efficient frontier: expected profit vs. risk aversion", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
plt.show()

## Sensitivity to alpha (the tail quantile)

The `beta` axis above is normalized by each alpha's own `C_loose`, so a chart built that way would look identical for every alpha: profit and CVaR are both exactly proportional to `x` here (`Loss_s(x) = x * (-price_s)`), so the tail ranking never changes with `x`, and normalizing hides any alpha effect.

To actually see the effect of alpha, we fix a common, absolute risk budget `C` on the x-axis and sweep it for several values of alpha. A higher alpha means we look at a smaller, more extreme slice of the tail (e.g. `alpha = 0.95` looks at only the worst 5% of scenarios instead of the worst 40%); the same absolute budget `C` then buys less investment, because the extreme tail is worse per unit invested.

In [ ]:
def optimal_investment(C, alpha):
    """Solve the CVaR-constrained investment problem for a given absolute risk budget C and confidence level alpha."""
    mp = gb.Model()
    mp.Params.OutputFlag = 0
    xp = mp.addVar(name="investment", lb=0, ub=10)
    zetap = mp.addVar(lb=-gb.GRB.INFINITY)
    up = mp.addVars(n_scenarios, lb=0)
    lossp = [-(xp * scenarios[s]) for s in range(n_scenarios)]
    mp.addConstrs((up[s] >= lossp[s] - zetap for s in range(n_scenarios)))
    cvarp = zetap + (1 / (1 - alpha)) * gb.quicksum(probability * up[s] for s in range(n_scenarios))
    mp.addConstr(cvarp <= C)
    mp.setObjective((xp * scenarios * probability).sum(), sense=gb.GRB.MAXIMIZE)
    mp.optimize()
    return xp.X


alphas = [0.6, 0.7, 0.8, 0.9, 0.95]
alpha_colors = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]  # sequential blue, light -> dark
C_grid = np.linspace(0, 200, 21)

alpha_profit = {
    alpha_i: [optimal_investment(C, alpha_i) * scenarios.mean() for C in C_grid]
    for alpha_i in alphas
}

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for alpha_i, color in zip(alphas, alpha_colors):
    ax.plot(C_grid, alpha_profit[alpha_i], color=color, linewidth=1.8, marker="o", markersize=3.5,
            label=f"alpha = {alpha_i:.0%}", zorder=3)

ax.axhline(profit_A.mean(), color=GRAY, linestyle="--", linewidth=1, zorder=1)
ax.set_xlabel("Risk budget, C (absolute, same units as profit)")
ax.set_ylabel("Expected profit")
ax.set_title("Sensitivity: expected profit vs. risk budget, by CVaR confidence level", loc="left", fontsize=11)
ax.legend(frameon=False, loc="lower right", title="Tail quantile")
fig.tight_layout()
plt.show()

## Takeaways

- Without CVaR, the optimizer only looks at the expected value, so it takes the maximum position (`x* = 10`) even though the worst scenario loses 200.
- Adding a CVaR constraint on the worst 20% of scenarios forces the optimizer to also look at the tail, not just the average. It scales the investment down (`x* = 4.4`), which shrinks the whole distribution (both the loss and the gain side), at the cost of a lower expected profit (181 vs 411).
- The risk budget `C` traces out a trade-off: a looser `C` moves the decision back toward the risk-neutral case, a tighter `C` moves it toward the safest possible choice (`x* = 0`, no exposure, no loss, no gain).
- The quantile `alpha` matters too: it changes how much budget `C` is needed to reach the same expected profit. A stricter alpha (closer to 1) looks only at the most extreme scenarios, so it takes a much larger risk budget before the investment decision is allowed to grow.